# SSS forecast evaluation (2023)

Daily RMSE / MAE / bias / correlation between the 4DVarNet SSS reconstruction and a reference SSS dataset, plus global error distribution and spatial error maps.

In [ ]:
import sys
sys.path.append('../..')  # adjust so contrib/ is importable, run from contrib/ose_pipeline/

import numpy as np
import matplotlib.pyplot as plt

from contrib.ose_pipeline.sss_eval import eval_sss_daily, summary_stats, load_aligned_fields

## Config — adjust paths/variables/leadtimes to match your run

In [ ]:
REC_PATHS = '/Odyssey/public/glorys/rec/glorys4_global_1patch_SSS_UNet_L3_NRT_AnomalyCLIMATO_2010_2019_MSEGradLoss_training_ose/nrt_sla/test_data_{}.nc'
LEADTIMES = range(11, 21)  # check which test_data_{leadtime}.nc files actually exist in REC_PATHS folder
REF_PATH = '/Odyssey/public/SALINITY_L3/NRT/SSS-L3-2010_2023_asc_desc_averaged_ANOMALY_CLIMATO_f32_QC_controled_flagged.nc'
REC_VAR = 'sos'
REF_VAR = 'sss_anomaly'
YEAR = 2023

LON_MIN, LON_MAX = -180., 180.
LAT_MIN, LAT_MAX = -83., 83.

## Daily metrics

In [ ]:
df = eval_sss_daily(
    rec_paths=REC_PATHS,
    leadtimes=LEADTIMES,
    ref_path=REF_PATH,
    rec_var=REC_VAR,
    ref_var=REF_VAR,
    year=YEAR,
    lon_min=LON_MIN, lon_max=LON_MAX, lat_min=LAT_MIN, lat_max=LAT_MAX,
    output_csv='sss_daily_metrics_2023.csv',
)
df.head()

In [ ]:
summary_stats(df)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

df['rmse'].plot(ax=axes[0], color='red')
axes[0].set_ylabel('RMSE')

df['mae'].plot(ax=axes[1], color='orange')
axes[1].set_ylabel('MAE')

df['bias'].plot(ax=axes[2], color='purple')
axes[2].axhline(0, color='k', lw=0.5)
axes[2].set_ylabel('Bias')

df['corr'].plot(ax=axes[3], color='green')
axes[3].set_ylabel('Correlation')
axes[3].set_xlabel('Date')

plt.suptitle('Daily SSS evaluation metrics — 2023')
plt.tight_layout()
plt.show()

## Global error distribution

Load the full aligned reconstruction / reference / difference fields (all days, all pixels) to look at the global error histogram, not just daily summary stats.

In [ ]:
rec, ref, diff = load_aligned_fields(
    rec_paths=REC_PATHS,
    leadtimes=LEADTIMES,
    ref_path=REF_PATH,
    rec_var=REC_VAR,
    ref_var=REF_VAR,
    year=YEAR,
    lon_min=LON_MIN, lon_max=LON_MAX, lat_min=LAT_MIN, lat_max=LAT_MAX,
)
diff

In [ ]:
errors = diff.values.ravel()
errors = errors[np.isfinite(errors)]

plt.figure(figsize=(8, 5))
plt.hist(errors, bins=200, color='steelblue')
plt.axvline(0, color='k', lw=0.8)
plt.xlabel('Error (rec - ref)')
plt.ylabel('Count')
plt.title(f'Global SSS error distribution — {YEAR}\n'
          f'mean={errors.mean():.4f}, std={errors.std():.4f}')
plt.show()

## Spatial error maps (time-averaged)

In [ ]:
mean_error_map = diff.mean(dim='time')
std_error_map = diff.std(dim='time')
rmse_map = np.sqrt((diff ** 2).mean(dim='time'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

mean_error_map.plot(ax=axes[0], cmap='RdBu_r', center=0)
axes[0].set_title('Mean error (bias) map')

std_error_map.plot(ax=axes[1], cmap='viridis')
axes[1].set_title('Std error map')

rmse_map.plot(ax=axes[2], cmap='viridis')
axes[2].set_title('RMSE map')

plt.tight_layout()
plt.show()